<a href="https://colab.research.google.com/github/2024meb1326-eng/Stress_Prediction-/blob/main/code/Ques_8_B).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras.layers import Dense, Input, Concatenate, Reshape, Flatten, Lambda
from keras.regularizers import l2
import keras.ops as kappa
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

np.random.seed(53)
tf.random.set_seed(53)

initializer1 = keras.initializers.HeNormal()

print("Loading cord.txt ...")
raw_cord_data = np.loadtxt("cord.txt")

full_cord_data = raw_cord_data[:, -2:].astype(np.float32)
N_total_nodes = full_cord_data.shape[0]

target_nodes = 600
slice_indices = np.linspace(0, N_total_nodes - 1, target_nodes, dtype=int)

cord_data = full_cord_data[slice_indices, :]
N_space = cord_data.shape[0]
print(f"Downsampled mesh from {N_total_nodes} nodes → {N_space} uniformly distributed nodes.")

print("Loading output.xlsx ...")
df = pd.read_excel("output.xlsx", header=0)
X_geom = df.values.astype(np.float32)
N_geom = X_geom.shape[0]
print("X_geom shape:", X_geom.shape)

print("Loading full spatial stress files ...")
stress_files = sorted(
    glob.glob(os.path.join("stress", "*.txt")),
    key=lambda p: int(os.path.splitext(os.path.basename(p))[0].split("_")[1])
)

Y_stress_field = np.zeros((N_geom, N_space), dtype=np.float32)

for i, fpath in enumerate(stress_files):
    data = np.loadtxt(fpath)
    Y_stress_field[i, :] = data[slice_indices, 0]
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(stress_files)} files loaded ...")

print("Y_stress_field shape:", Y_stress_field.shape)


scaler_geom = StandardScaler()
X_geom_scaled = scaler_geom.fit_transform(X_geom)

X_space_min = cord_data.min(axis=0)
X_space_max = cord_data.max(axis=0)
X_space_scaled = 2.0 * (cord_data - X_space_min) / (X_space_max - X_space_min + 1e-8) - 1.0

scaler_Y = StandardScaler()
Y_flat = Y_stress_field.reshape(-1, 1)
Y_flat_scaled = scaler_Y.fit_transform(Y_flat)
Y_scaled_field = Y_flat_scaled.reshape(N_geom, N_space)

indices = np.arange(N_geom)
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=40)

X_geom_train, X_geom_test = X_geom_scaled[train_idx], X_geom_scaled[test_idx]
Y_train, Y_test = Y_scaled_field[train_idx], Y_scaled_field[test_idx]

print(f"Train structures: {X_geom_train.shape[0]} | Test structures: {X_geom_test.shape[0]}")

def positional_encoding(x):
    """ Project low dimensional coordinates into multiple sin/cos frequency bands """
    bands = [1.0, 2.0, 4.0, 8.0, 16.0]
    encodings = [x]
    for b in bands:
        encodings.append(kappa.sin(x * b))
        encodings.append(kappa.cos(x * b))
    return Concatenate(axis=-1)(encodings)

tf.keras.backend.clear_session()

input_geom = Input(shape=(X_geom_train.shape[1],), name='Geometry_Input')
x_geo = Dense(128, activation='relu', kernel_initializer=initializer1, kernel_regularizer=l2(1e-4))(input_geom)
x_geo = Dense(64, activation='relu', kernel_initializer=initializer1, kernel_regularizer=l2(1e-4))(x_geo)
geo_embedding = Dense(32, activation='relu', name='Geo_Embedding')(x_geo)

input_space = Input(shape=(N_space, 2), name='Spatial_Input')
encoded_space = Lambda(positional_encoding, name='Positional_Encoding_Layer')(input_space)

x_spa = Dense(64, activation='relu', kernel_initializer=initializer1)(encoded_space)
x_spa = Dense(32, activation='relu', kernel_initializer=initializer1)(x_spa)
spa_embedding = Dense(32, activation='relu', name='Spatial_Embedding')(x_spa)

geo_expanded = Reshape((1, 32))(geo_embedding)
geo_tiled = kappa.tile(geo_expanded, [1, N_space, 1])

fused = Concatenate(axis=-1)([geo_tiled, spa_embedding])

x = Dense(64, activation='relu', kernel_initializer=initializer1, kernel_regularizer=l2(1e-4))(fused)
x = Dense(32, activation='relu', kernel_initializer=initializer1, kernel_regularizer=l2(1e-4))(x)
x = Dense(16, activation='relu', kernel_initializer=initializer1)(x)
output_stress = Dense(1, name='Stress_Output')(x)

output_stress = Flatten()(output_stress)

model2 = keras.Model(inputs=[input_geom, input_space], outputs=output_stress, name="Two_Stream_Model")
model2.summary()


model2.compile(optimizer=keras.optimizers.Adam(learning_rate=5e-4), loss='mse', metrics=['mae'])

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1)
reduce_lr  = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-6, verbose=1)

X_space_train_input = np.repeat(X_space_scaled[np.newaxis, :, :], X_geom_train.shape[0], axis=0)
X_space_test_input  = np.repeat(X_space_scaled[np.newaxis, :, :], X_geom_test.shape[0], axis=0)

history = model2.fit(
    [X_geom_train, X_space_train_input], Y_train,
    epochs=150,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr]
)

print("\nEvaluating model predictions...")
Y_pred_train_scaled = model2.predict([X_geom_train, X_space_train_input])
Y_pred_test_scaled  = model2.predict([X_geom_test, X_space_test_input])

Y_pred_train_flat = scaler_Y.inverse_transform(Y_pred_train_scaled.reshape(-1, 1)).ravel()
Y_train_orig_flat = scaler_Y.inverse_transform(Y_train.reshape(-1, 1)).ravel()

Y_pred_test_flat  = scaler_Y.inverse_transform(Y_pred_test_scaled.reshape(-1, 1)).ravel()
Y_test_orig_flat  = scaler_Y.inverse_transform(Y_test.reshape(-1, 1)).ravel()

r2_train   = r2_score(Y_train_orig_flat, Y_pred_train_flat)
mae_train  = mean_absolute_error(Y_train_orig_flat, Y_pred_train_flat)
rmse_train = np.sqrt(mean_squared_error(Y_train_orig_flat, Y_pred_train_flat))

r2_test    = r2_score(Y_test_orig_flat, Y_pred_test_flat)
mae_test   = mean_absolute_error(Y_test_orig_flat, Y_pred_test_flat)
rmse_test  = np.sqrt(mean_squared_error(Y_test_orig_flat, Y_pred_test_flat))

print("\n========== TRAIN SET METRICS (PART 2 - OPTIMIZED) ==========")
print(f"Global R² Score : {r2_train:.5f}")
print(f"Global MAE      : {mae_train:.5f}")
print(f"Global RMSE     : {rmse_train:.5f}")

print("\n========== TEST SET METRICS (PART 2 - OPTIMIZED) ==========")
print(f"Global R² Score : {r2_test:.5f}")
print(f"Global MAE      : {mae_test:.5f}")
print(f"Global RMSE     : {rmse_test:.5f}")

plt.figure(figsize=(6, 6))
plt.scatter(Y_test_orig_flat[::10], Y_pred_test_flat[::10], s=5, alpha=0.3, color='teal', label='Sampled Nodes')
lims = [min(Y_test_orig_flat.min(), Y_pred_test_flat.min()), max(Y_test_orig_flat.max(), Y_pred_test_flat.max())]
plt.plot(lims, lims, 'r--', label='Perfect Agreement')
plt.xlabel('Actual Node Stress')
plt.ylabel('Predicted Node Stress')
plt.title(f'Spatial Node Parity Plot\nTest R² = {r2_test:.4f}')
plt.legend()
plt.tight_layout()
plt.savefig('spatial_parity_test.png', dpi=150)
plt.show()

Y_test_reconstructed_matrices = Y_test_orig_flat.reshape(X_geom_test.shape[0], N_space)
Y_pred_reconstructed_matrices = Y_pred_test_flat.reshape(X_geom_test.shape[0], N_space)

random_sample_idx = np.random.randint(0, X_geom_test.shape[0])

true_field = Y_test_reconstructed_matrices[random_sample_idx, :]
pred_field = Y_pred_reconstructed_matrices[random_sample_idx, :]
error_field = np.abs(true_field - pred_field)

sample_r2 = r2_score(true_field, pred_field)
sample_rmse = np.sqrt(mean_squared_error(true_field, pred_field))

x_coords = cord_data[:, 0]
y_coords = cord_data[:, 1]

vmin_val = min(true_field.min(), pred_field.min())
vmax_val = max(true_field.max(), pred_field.max())

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
fig.suptitle(f'Stress Field — Sample {random_sample_idx}  (Ensemble R²={sample_r2:.4f}  RMSE={sample_rmse:.2f})', fontsize=14)

sc1 = axes[0].scatter(x_coords, y_coords, c=true_field, cmap='jet', vmin=vmin_val, vmax=vmax_val, s=25)
axes[0].set_title('True stress field')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
fig.colorbar(sc1, ax=axes[0], label='Von Mises stress')

sc2 = axes[1].scatter(x_coords, y_coords, c=pred_field, cmap='jet', vmin=vmin_val, vmax=vmax_val, s=25)
axes[1].set_title('Predicted (ensemble)')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
fig.colorbar(sc2, ax=axes[1], label='Von Mises stress')

sc3 = axes[2].scatter(x_coords, y_coords, c=error_field, cmap='hot_r', s=25)
axes[2].set_title('Absolute error')
axes[2].set_xlabel('x')
axes[2].set_ylabel('y')
fig.colorbar(sc3, ax=axes[2], label='|error|')

plt.tight_layout()
plt.savefig(f'stress_field_sample{random_sample_idx}.png', dpi=150)
plt.show()

print(f"\nOptimization complete! Visualizing outputs saved successfully.")